In [1]:
pip show paddleocr

Name: paddleocr
Version: 3.3.2
Summary: Awesome multilingual OCR and document parsing toolkits based on PaddlePaddle
Home-page: https://github.com/PaddlePaddle/PaddleOCR
Author: 
Author-email: PaddlePaddle <paddleocr@baidu.com>
License: Apache License 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: paddlex, PyYAML, requests, typing-extensions
Required-by: 


In [1]:
! pip install "langchain<0.1.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might ne

In [2]:
!pip install paddleocr
!pip install paddlepaddle
!pip install langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.7/67.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.0/87.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/30

In [3]:
import cv2
import time
import numpy as np
import re
from datetime import datetime, timedelta
from paddleocr import PaddleOCR

# =========================================================
# 1. OCR 모델 초기화 (속도 최적화 설정)
# =========================================================
# 프로그램 시작 시 한 번만 로딩합니다.
print("🔄 모델을 메모리에 로딩 중입니다... (최초 1회)")
ocr = PaddleOCR(
    lang='korean',
    use_angle_cls=False,     # [속도 핵심] 문서가 회전되지 않았다면 False 추천
    enable_mkldnn=True,      # [속도 핵심] CPU 가속 활성화
)
print("✅ 모델 로딩 완료!")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# =========================================================
# 3. 메인 파싱 함수
# =========================================================
def parse_chat_image(img_path):
    start_time = time.time()

    # 3-1. 이미지 읽기 (한글 경로 대응)
    try:
        img_array = np.fromfile(img_path, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    except Exception:
        return "Error: 이미지를 읽을 수 없습니다."

    if img is None: return "Error: 이미지가 없습니다."

    # 3-2. 리사이징 (속도 최적화: 960px)
    h, w, _ = img.shape
    max_side = 1080
    if max(h, w) > max_side:
        scale = max_side / max(h, w)
        img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)

    current_width = img.shape[1]
    center_x = current_width / 2

    # 3-3. OCR 수행
    result = ocr.ocr(img)
    print(f"🚀 OCR 처리 소요시간: {time.time() - start_time:.4f}초")
    return result

    if not result or not result[0]:
        return "텍스트를 찾을 수 없습니다."

# =========================================================
# 4. 실행
# =========================================================
if __name__ == '__main__':
    image_file = 'screenshot_2.png'  # 이미지 경로

    print(f"📂 분석 시작: {image_file}")
    result_text = parse_chat_image(image_file)

    print("\n" + "="*50)
    print("           [ 최종 변환 결과 ]")
    print("="*50)
    print(result_text)

In [ ]:
import numpy as np
import re
from datetime import datetime, timedelta

# =========================================================
# 1. 유틸리티 함수
# =========================================================
def clean_text(text):
    """특수문자 및 노이즈 제거"""
    text = re.sub(r"[|│\\]", "", text)
    return text.strip()

def is_noise(text):
    """기본적인 시스템 메시지 노이즈 제거"""
    if not text: return True

    exclude_keywords = ["공유하기", "들어왔습니다", "나갔습니다", "초대했습니다"]
    for keyword in exclude_keywords:
        if keyword in text:
            return True

    system_patterns = [
        r"들어왔습니다\.$", r"나갔습니다\.$",
        r"초대했습니다\.$", r"^삭제된 메시지\.$",
        r".*기프티콘을 보냈습니다.*", r".*송금했습니다.*", r"^톡게시판.*",
    ]
    for pattern in system_patterns:
        if re.match(pattern, text): return True

    ui_keywords = {
        "사진", "동영상", "음성메시지", "보이스톡", "페이스톡", "라이브톡",
        "선물하기", "송금", "정산하기", "프로필 보기", "공지 등록", "좋아요", "공감",
        "안읽음", "MY", "채팅방", "메뉴", "전송", "읽음"
    }
    if text in ui_keywords: return True
    if text.replace(':', '').isdigit() and len(text) < 3: return True
    return False

def format_time(ts_str):
    """'오후9:24' -> '21:24' 포맷 변환"""
    ts_str = ts_str.replace(" ", "")
    final_time = ts_str
    try:
        is_pm = "오후" in ts_str
        is_am = "오전" in ts_str
        match = re.search(r"(\d{1,2})[:\.\,](\d{2})", ts_str)
        if not match and (is_pm or is_am):
            match = re.search(r"(\d{1,2})(\d{2})$", ts_str)
        if match:
            hour, minute = int(match.group(1)), int(match.group(2))
            if is_pm and hour != 12: hour += 12
            if is_am and hour == 12: hour = 0
            final_time = f"{hour:02}:{minute:02}"
    except: pass
    return final_time

def parse_kakao_dict(ocr_result, image_width=720, image_height=None):
    if 'rec_texts' not in ocr_result or 'rec_polys' not in ocr_result:
        return "Error: 데이터 형식이 올바르지 않습니다."

    texts = ocr_result['rec_texts']
    polys = ocr_result['rec_polys']
    center_x = image_width / 2

    # 이미지 높이 추정 (입력이 없을 경우)
    if image_height is None:
        all_ys = []
        for p in polys:
            np_p = np.array(p)
            all_ys.extend(np_p[:, 1])
        image_height = max(all_ys) if all_ys else 2000

    # -------------------------------------------------------------
    # [수정 2] 상단 영역(헤더) 및 공지사항 필터링 기준 설정
    # -------------------------------------------------------------
    # 기본적으로 상단 1/8은 무시 (채팅방 이름, 메뉴 등)
    y_start_threshold = image_height / 8

    # 공지사항 감지용 임계값 (상단 30% 지점까지만 검사)
    notice_search_limit = image_height * 0.2

    # 정규식 정의
    time_regexs = [
        re.compile(r".*[\d]{1,2}:[\d]{2}$"),
        re.compile(r".*[\d]{1,2}\.[\d]{2}$"),
        re.compile(r".*[\d]{1,2},[\d]{2}$"),
        re.compile(r".*오[전후]\s*\d{3,4}$")
    ]
    date_regex = re.compile(r"^20\d{2}[.년]\s*\d{1,2}[.월]\s*\d{1,2}[.일]?.*")

    # [수정 1] 스크롤 타임스탬프 정규식 ("11. 11. 화" 형태, 띄어쓰기 유연하게)
    # 예: 11.11.화, 11. 11. 화
    scroll_date_regex = re.compile(r"^\d{1,2}[\.\s]+\d{1,2}[\.\s]+[월화수목금토일]")

    # -------------------------------------------------------------
    # 1차 패스: 공지사항 위치 감지 및 y_start_threshold 조정
    # -------------------------------------------------------------
    for text, poly in zip(texts, polys):
        try:
            np_poly = np.array(poly)
            y_max = np.max(np_poly[:, 1]) # 텍스트 박스 하단
            y_center = (np.min(np_poly[:, 1]) + y_max) / 2
            x_left = np.min(np_poly[:, 0])
        except: continue

        # 상단 영역(30% 이내)에 있고, 왼쪽 끝(20px 미만)에 붙어있는 텍스트가 있다면
        # 공지사항이나 배너로 간주하고 시작 기준점을 그 아래로 내림
        if y_center < notice_search_limit and x_left < 50:
            if y_max > y_start_threshold:
                y_start_threshold = y_max + 5 # 여유분 5px

    raw_items = []

    # -------------------------------------------------------------
    # 2차 패스: 데이터 필터링 및 수집
    # -------------------------------------------------------------
    for text, poly in zip(texts, polys):
        text = clean_text(text)
        if not text or is_noise(text): continue

        try:
            np_poly = np.array(poly)
            y_center = (np.min(np_poly[:, 1]) + np.max(np_poly[:, 1])) / 2
            x_left = np.min(np_poly[:, 0])
            x_right = np.max(np_poly[:, 0])
            x_center = (x_left + x_right) / 2
        except: continue

        # [수정 2 적용] 상단 영역(헤더+공지) 무시
        if y_center < y_start_threshold:
            continue

        # [수정 1 적용] 우측 스크롤 타임스탬프 무시
        # 패턴이 일치하고, 화면 오른쪽에 있다면 삭제
        if scroll_date_regex.search(text):
            continue

        is_timestamp = any(r.match(text) for r in time_regexs)
        is_date = bool(date_regex.match(text))

        raw_items.append({
            'text': text, 'y_center': y_center, 'x_left': x_left, 'x_right': x_right,
            'is_timestamp': is_timestamp, 'is_date': is_date
        })

    if not raw_items: return "텍스트가 없습니다."

    # Y축 정렬
    raw_items.sort(key=lambda x: x['y_center'])

    # "에게 답장" 제거 로직 (이전과 동일)
    indices_to_remove = set()
    for i in range(len(raw_items)):
        if "에게 답장" in raw_items[i]['text'] or "에게답장" in raw_items[i]['text']:
            indices_to_remove.add(i)
            if i + 1 < len(raw_items):
                next_item = raw_items[i+1]
                if not next_item['is_timestamp'] and not next_item['is_date']:
                    if (next_item['y_center'] - raw_items[i]['y_center']) < 100:
                        indices_to_remove.add(i+1)

    processed_items = [item for i, item in enumerate(raw_items) if i not in indices_to_remove]

    # 줄 그룹화
    Y_TOLERANCE = 50
    grouped_lines = []
    if processed_items:
        curr_items = [processed_items[0]]
        curr_y = processed_items[0]['y_center']
        for item in processed_items[1:]:
            if abs(item['y_center'] - curr_y) < Y_TOLERANCE:
                curr_items.append(item)
            else:
                _process_and_save_groups(grouped_lines, curr_items)
                curr_items = [item]
                curr_y = item['y_center']
        _process_and_save_groups(grouped_lines, curr_items)

    # 날짜 초기화
    detected_dates = []
    for line in grouped_lines:
        if line['type'] == 'date':
            text = " ".join([i['text'] for i in line['items']])
            match = re.search(r"(\d{4})[.년]\s*(\d{1,2})[.월]\s*(\d{1,2})", text)
            if match:
                y, m, d = map(int, match.groups())
                detected_dates.append(datetime(y, m, d))

    if detected_dates:
        earliest_date = min(detected_dates)
        start_date = earliest_date - timedelta(days=1)
        current_date = f"{start_date.year}. {start_date.month}. {start_date.day}."
    else:
        current_date = "2000. 1. 1."

    # -------------------------------------------------------------
    # 메인 변환 루프 (발화자 추론 로직 수정)
    # -------------------------------------------------------------
    final_chat = []
    current_turn_lines = []

    for line in grouped_lines:
        if line['type'] == 'date':
            raw_date = " ".join([i['text'] for i in line['items']])
            match = re.search(r"(\d{4})[.년]\s*(\d{1,2})[.월]\s*(\d{1,2})", raw_date)
            if match:
                current_date = f"{match.group(1)}. {match.group(2)}. {match.group(3)}."

        elif line['type'] == 'text':
            current_turn_lines.append(line)

        elif line['type'] == 'timestamp':
            if not current_turn_lines: continue

            raw_time = " ".join([i['text'] for i in line['items']])
            time_str = format_time(raw_time)
            full_ts = f"{current_date} {time_str}"

            first_line = current_turn_lines[0]
            first_min_x = min(item['x_left'] for item in first_line['items'])
            first_max_x = max(item['x_right'] for item in first_line['items'])
            first_center_x = (first_min_x + first_max_x) / 2

            speaker = ""
            messages = []

            if first_center_x > center_x:
                # [오른쪽] 나
                speaker = "나"
                for l in current_turn_lines:
                    messages.append(" ".join([i['text'] for i in l['items']]))
            else:
                # [왼쪽] 상대방

                # [수정 3] 발화자 이름 유무 확인 로직
                # 첫 줄이 이름인지 메시지인지 판단 (길이와 위치 기반)
                first_text = " ".join([i['text'] for i in first_line['items']])

                # 조건: 길이가 짧고(15자 미만), 텍스트 박스가 아주 왼쪽은 아니지만(프로필 옆),
                # 그 다음 줄이 존재한다면 이름일 확률이 높음.
                # 하지만 이름이 없는 경우(연속 대화 중 잘림 등)를 대비해 기본값 설정

                # 간단한 휴리스틱: 첫 줄이 메시지처럼 길면(15자 이상) 이름이 생략된 것으로 간주
                # 또는 첫 줄만 있고 메시지가 없다면(사진 등) 이름일 수 있음 -> 이는 상황에 따라 다름

                is_likely_name = len(first_text) < 15 and len(current_turn_lines) > 1

                if is_likely_name:
                    speaker = first_text
                    start_idx = 1 # 첫 줄은 이름이므로 메시지는 두 번째 줄부터
                else:
                    speaker = "상대방" # 이름이 없다고 판단
                    start_idx = 0 # 첫 줄부터 메시지

                for l in current_turn_lines[start_idx:]:
                    messages.append(" ".join([i['text'] for i in l['items']]))

            full_message = " ".join(messages)

            if full_message.strip():
                final_chat.append(f"{full_ts}, {speaker} : {full_message}")
            elif speaker != "나" and not full_message.strip():
                 # 상대방 메시지가 비어있다면 (사진/이모티콘 등)
                 final_chat.append(f"{full_ts}, {speaker} : (사진/이모티콘)")

            current_turn_lines = []

    return "\n".join(final_chat)

def _process_and_save_groups(grouped_lines, items):
    items.sort(key=lambda x: x['y_center'])
    sub_lines = []
    if items:
        curr_sub = [items[0]]
        for item in items[1:]:
            if (item['y_center'] - curr_sub[-1]['y_center']) > 12:
                sub_lines.append(curr_sub)
                curr_sub = [item]
            else:
                curr_sub.append(item)
        sub_lines.append(curr_sub)

    for line_items in sub_lines:
        line_items.sort(key=lambda x: x['x_left'])
        texts = [i for i in line_items if not i['is_timestamp'] and not i['is_date']]
        times = [i for i in line_items if i['is_timestamp']]
        dates = [i for i in line_items if i['is_date']]

        if texts: grouped_lines.append({'type': 'text', 'items': texts})
        if times: grouped_lines.append({'type': 'timestamp', 'items': times})
        if dates: grouped_lines.append({'type': 'date', 'items': dates})

if __name__ == "__main__":
    # 데이터 샘플 (텍스트 순서 테스트)
    # 실제 사용 시에는 ocr_result를 여기에 넣으세요
    print("▶ 파싱 시작...")
    parsed_log = parse_kakao_dict(result_text[0], image_width = 884)

    print("\n" + "="*30)
    print("      [변환 결과]")
    print("="*30)
    print(parsed_log)
    pass